# 10 · Tucker decomposition on real data / Descomposición de Tucker con datos reales

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/10-tucker-decomposition.ipynb)

*Part IV · exercise · 15 min*

This notebook answers one practical question:

> **How can we compress a tensor while keeping the meaning of its different axes?**

We will build a real tensor from New York taxi trips:

`pickup borough × dropoff borough × hour`

and use Tucker/HOSVD to compress the three modes separately.

> 🇪🇸 Este cuaderno responde una pregunta práctica:
>
> **¿Cómo podemos comprimir un tensor conservando el significado de sus distintos ejes?**
>
> Construiremos un tensor real de viajes en taxi:
>
> `distrito de origen × distrito de destino × hora`
>
> y usaremos Tucker/HOSVD para comprimir los tres modos por separado.

## What you will be able to do / Lo que podrás hacer

- Turn a flat table of real taxi trips into an order-3 tensor.
- Explain **mode, unfolding, factor matrix, core tensor, multilinear rank, reconstruction error, and compression** in plain language.
- Unfold the same tensor along pickup, dropoff, and hour modes.
- Compute HOSVD using SVD on each unfolding.
- Build and reconstruct a Tucker model with `np.einsum`.
- Change the three Tucker ranks independently and see the error–storage trade-off.
- Interpret learned temporal components against the real hourly taxi counts.

> 🇪🇸
>
> - Convertir una tabla plana de viajes reales en un tensor de orden 3.
> - Explicar en lenguaje sencillo **modo, unfolding, matriz de factores, tensor núcleo, rango multilineal, error de reconstrucción y compresión**.
> - Desplegar el mismo tensor por origen, destino y hora.
> - Calcular HOSVD usando SVD sobre cada unfolding.
> - Construir y reconstruir Tucker con `np.einsum`.
> - Cambiar independientemente los tres rangos Tucker y observar el compromiso entre error y almacenamiento.
> - Interpretar componentes temporales aprendidos comparándolos con los conteos reales por hora.

## Start with an everyday analogy / Empecemos con una analogía cotidiana

Imagine a **3D spreadsheet**.

A normal spreadsheet has:

`rows × columns`

Our taxi tensor has:

`pickup × dropoff × hour`

So every cell answers a concrete question:

> **How many trips started here, ended there, at this hour?**

For example:

`T[i, j, h]`

is one real trip count.

### What does Tucker do? / ¿Qué hace Tucker?

Imagine that many pickup borough patterns look similar, many dropoff patterns look similar, and several hours share similar traffic structure.

Instead of storing every detail independently, Tucker learns:

1. a small set of **pickup patterns**;
2. a small set of **dropoff patterns**;
3. a small set of **hour patterns**;
4. a small **core tensor** that says how those patterns interact.

> 🇪🇸 Imagina una **hoja de cálculo 3D**.
>
> Cada celda del tensor responde:
>
> **¿Cuántos viajes empezaron aquí, terminaron allá y ocurrieron a esta hora?**
>
> Tucker intenta resumir el tensor usando pocos patrones de origen, pocos patrones de destino, pocos patrones horarios y un **núcleo pequeño** que describe cómo interactúan.

## Seven words before the decomposition / Siete palabras antes de la descomposición

| Term / Término | Plain meaning / Significado sencillo |
|---|---|
| **Mode / Modo** | one semantic axis of a tensor / un eje semántico del tensor |
| **Unfolding / Desplegado** | rearrange a tensor as a matrix while keeping all entries / reorganizar un tensor como matriz conservando todas las entradas |
| **SVD basis / Base SVD** | important directions found in one unfolding / direcciones importantes encontradas en un unfolding |
| **Factor matrix / Matriz de factores** | the retained basis vectors for one mode / los vectores de base conservados para un modo |
| **Core tensor / Tensor núcleo** | a small tensor describing how retained mode patterns interact / tensor pequeño que describe cómo interactúan los patrones conservados |
| **Multilinear rank / Rango multilineal** | one retained rank per mode, e.g. `(2,2,3)` / un rango conservado por modo, por ejemplo `(2,2,3)` |
| **Reconstruction / Reconstrucción** | approximate tensor rebuilt from core + factors / tensor aproximado reconstruido desde núcleo + factores |

### One sentence to remember / Una frase para recordar

> **SVD compresses one matrix view; Tucker uses one compressed basis for each tensor mode.**

> 🇪🇸
>
> **SVD comprime una vista matricial; Tucker usa una base comprimida para cada modo del tensor.**

## Setup / Preparación

Run this first.

We use the real New York taxi table distributed through the Seaborn example-data repository.

The original table contains **6,433 rows**. We keep only trips with:

- a pickup borough,
- a dropoff borough,
- a valid pickup time.

Then we count usable trips into:

`T[pickup, dropoff, hour]`

> 🇪🇸 Ejecuta primero esta celda.
>
> Usamos la tabla real de taxis de Nueva York distribuida en los datos de ejemplo de Seaborn.
>
> La tabla original contiene **6.433 filas**. Conservamos solo los viajes con distrito de origen, distrito de destino y hora de recogida válidos, y los contamos dentro de:
>
> `T[origen, destino, hora]`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

TAXIS = (
    "https://raw.githubusercontent.com/mwaskom/"
    "seaborn-data/master/taxis.csv"
)

taxis = pd.read_csv(TAXIS)

def unfold(T, axis):
    # Move the selected mode to the front, then flatten all other modes.
    return np.moveaxis(T, axis, 0).reshape(T.shape[axis], -1)

def hosvd_bases(T):
    # One left-singular-vector basis per mode unfolding.
    return [
        np.linalg.svd(
            unfold(T, axis),
            full_matrices=False,
        )[0]
        for axis in range(T.ndim)
    ]

taxis["pickup_dt"] = pd.to_datetime(
    taxis["pickup"],
    errors="coerce",
)

taxis["hour"] = taxis["pickup_dt"].dt.hour

sub = taxis.dropna(
    subset=[
        "pickup_borough",
        "dropoff_borough",
        "hour",
    ]
).copy()

sub["hour"] = sub["hour"].astype(int)

pickup_names = sorted(
    sub["pickup_borough"].unique()
)

dropoff_names = sorted(
    sub["dropoff_borough"].unique()
)

pickup_index = {
    name: i
    for i, name in enumerate(pickup_names)
}

dropoff_index = {
    name: i
    for i, name in enumerate(dropoff_names)
}

T = np.zeros(
    (
        len(pickup_names),
        len(dropoff_names),
        24,
    ),
    dtype=float,
)

for (p, d, h), count in sub.groupby(
    [
        "pickup_borough",
        "dropoff_borough",
        "hour",
    ]
).size().items():
    T[
        pickup_index[p],
        dropoff_index[d],
        int(h),
    ] = float(count)

print("Taxi table rows / Filas de la tabla:", len(taxis))
print("Usable trips / Viajes utilizables:", int(T.sum()))
print("Tensor shape / Forma del tensor:", T.shape)
print("Order / Orden:", T.ndim)
print()
print("Pickup boroughs / Distritos de origen:", pickup_names)
print("Dropoff boroughs / Distritos de destino:", dropoff_names)
print()
print("EN: T[pickup, dropoff, hour] stores a real trip count.")
print("ES: T[origen, destino, hora] almacena un conteo real de viajes.")

## Why this matters / Por qué esto importa

If we flatten everything immediately, we lose the **visible separation between meanings**.

In the taxi tensor:

- mode 0 = pickup borough;
- mode 1 = dropoff borough;
- mode 2 = hour.

Tucker respects that structure by giving **each mode its own basis**.

### Why not call this “PCA on a tensor”? / ¿Por qué no llamarlo “PCA sobre un tensor”?

HOSVD uses SVD on each mode unfolding, so it is related to familiar low-dimensional matrix ideas.

But Tucker is better described as a **multilinear extension of truncated-SVD ideas** across several modes.

Each axis gets its own factor matrix, and a core tensor connects those mode-specific factors.

### Learning cycle / Ciclo de aprendizaje

Use:

**Predict → Run → Explain / Predice → Ejecuta → Explica**

Before each exercise ask:

1. What does every axis mean?
2. Which mode is being unfolded?
3. What shape should the matrix have?
4. Which ranks are being kept?
5. What information gets harder to reconstruct when a rank is reduced?

> 🇪🇸 Tucker conserva la separación semántica de los ejes porque cada modo obtiene su propia base.
>
> Es más preciso entender HOSVD/Tucker como una extensión multilineal de ideas de SVD truncada, no simplemente como “PCA convertido en tensor”.

### Interactive tensor-cell explorer / Explorador interactivo de celdas del tensor

Choose:

- pickup borough / distrito de origen;
- dropoff borough / distrito de destino;
- hour / hora.

The notebook will read one real tensor entry:

`T[i,j,h]`

> 🇪🇸 Elige origen, destino y hora. El cuaderno leerá una celda real del tensor `T[i,j,h]`.

In [ ]:
pickup_dropdown = widgets.Dropdown(
    options=[
        (name, i)
        for i, name in enumerate(pickup_names)
    ],
    value=0,
    description="Pickup / Origen:",
    style={"description_width": "110px"},
)

dropoff_dropdown = widgets.Dropdown(
    options=[
        (name, i)
        for i, name in enumerate(dropoff_names)
    ],
    value=0,
    description="Dropoff / Destino:",
    style={"description_width": "120px"},
)

hour_entry_slider = widgets.IntSlider(
    value=12,
    min=0,
    max=23,
    step=1,
    description="Hour / Hora:",
    continuous_update=False,
    style={"description_width": "95px"},
)

def inspect_tensor_entry(pickup, dropoff, hour):
    count = int(
        T[pickup, dropoff, hour]
    )

    print(
        f"T[{pickup}, {dropoff}, {hour}] = {count}"
    )
    print()
    print(
        "Pickup / Origen:",
        pickup_names[pickup],
    )
    print(
        "Dropoff / Destino:",
        dropoff_names[dropoff],
    )
    print("Hour / Hora:", hour)
    print("Trips / Viajes:", count)
    print()
    print("EN: this is one measured count stored at one tensor coordinate.")
    print("ES: este es un conteo medido almacenado en una coordenada del tensor.")

entry_output = widgets.interactive_output(
    inspect_tensor_entry,
    {
        "pickup": pickup_dropdown,
        "dropoff": dropoff_dropdown,
        "hour": hour_entry_slider,
    },
)

display(
    widgets.VBox([
        widgets.HBox([
            pickup_dropdown,
            dropoff_dropdown,
        ]),
        hour_entry_slider,
        entry_output,
    ])
)

## Exercise 1 — understand the tensor before compressing it / Ejercicio 1 — entiende el tensor antes de comprimirlo

Before decomposition, answer basic questions about the real observations.

### Predict first / Predice primero

If:

`T.shape = (P, D, 24)`

then:

- pickup unfolding should have `P` rows;
- dropoff unfolding should have `D` rows;
- hour unfolding should have `24` rows.

Every unfolding must contain exactly the same number of tensor entries.

> 🇪🇸 Antes de descomponer el tensor, comprende los datos reales.
>
> Si `T.shape = (P,D,24)`, el unfolding de origen debe tener `P` filas, el de destino `D` filas y el horario `24` filas.
>
> Todos los unfoldings deben conservar exactamente la misma cantidad de entradas.

In [ ]:
# TODO 1 / TAREA 1
#
# EN:
# 1. Print T.shape and T.sum().
# 2. Compute T.sum(axis=(0,1)).
# 3. Find the busiest hour.
# 4. Print unfold(T, axis).shape for axis 0, 1, and 2.
# 5. Explain what the rows mean in each unfolding.
#
# ES:
# 1. Imprime T.shape y T.sum().
# 2. Calcula T.sum(axis=(0,1)).
# 3. Encuentra la hora con mayor número de viajes.
# 4. Imprime unfold(T, axis).shape para los ejes 0, 1 y 2.
# 5. Explica qué significan las filas de cada unfolding.

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

print("T.shape / Forma de T:", T.shape)
print(
    "Total usable trips / Total viajes utilizables:",
    int(T.sum()),
)

by_hour = T.sum(
    axis=(0, 1)
)

busiest_hour = int(
    np.argmax(by_hour)
)

print()
print(
    "Busiest hour / Hora más ocupada:",
    busiest_hour,
)
print(
    "Trips at busiest hour / Viajes en esa hora:",
    int(by_hour[busiest_hour]),
)

mode_meanings = {
    0: (
        "one row per pickup borough",
        "una fila por distrito de origen",
    ),
    1: (
        "one row per dropoff borough",
        "una fila por distrito de destino",
    ),
    2: (
        "one row per hour",
        "una fila por hora",
    ),
}

print()

for axis in range(3):
    M = unfold(
        T,
        axis,
    )

    en, es = mode_meanings[axis]

    print(
        f"axis/eje {axis}: "
        f"shape/forma={M.shape} | "
        f"same entries/mismas entradas={M.size == T.size}"
    )
    print("  EN:", en)
    print("  ES:", es)

print()
print("EN: unfolding rearranges entries; it does not delete trip counts.")
print("ES: unfolding reorganiza las entradas; no elimina conteos de viajes.")

### Interactive hour explorer / Explorador interactivo por hora

Move **Hour / Hora**.

For each hour, you will see the real:

`pickup × dropoff`

count matrix.

> 🇪🇸 Mueve **Hora**. Para cada hora verás la matriz real `origen × destino`.

In [ ]:
hour_slider = widgets.IntSlider(
    value=busiest_hour,
    min=0,
    max=23,
    step=1,
    description="Hour / Hora:",
    continuous_update=False,
    style={"description_width": "95px"},
)

def show_hour(hour):
    matrix = T[:, :, hour]

    fig, ax = plt.subplots(
        figsize=(6.4, 4.8),
        constrained_layout=True,
    )

    im = ax.imshow(
        matrix,
        cmap="viridis",
        aspect="auto",
        vmin=0,
    )

    ax.set_xticks(
        range(len(dropoff_names))
    )
    ax.set_xticklabels(
        dropoff_names,
        rotation=35,
        ha="right",
        fontsize=8,
    )

    ax.set_yticks(
        range(len(pickup_names))
    )
    ax.set_yticklabels(
        pickup_names,
        fontsize=8,
    )

    ax.set_xlabel(
        "dropoff borough / destino"
    )
    ax.set_ylabel(
        "pickup borough / origen"
    )

    ax.set_title(
        f"Real taxi counts — hour {hour}\n"
        f"Conteos reales — hora {hour}"
    )

    fig.colorbar(
        im,
        ax=ax,
        label="trip count / viajes",
    )

    plt.show()

    print(
        "Total trips this hour / Viajes en esta hora:",
        int(matrix.sum()),
    )

    busiest_pair = np.unravel_index(
        np.argmax(matrix),
        matrix.shape,
    )

    print(
        "Largest origin→destination cell / "
        "Mayor celda origen→destino:"
    )
    print(
        pickup_names[busiest_pair[0]],
        "→",
        dropoff_names[busiest_pair[1]],
        "=",
        int(matrix[busiest_pair]),
    )

hour_output = widgets.interactive_output(
    show_hour,
    {"hour": hour_slider},
)

display(
    widgets.VBox([
        hour_slider,
        hour_output,
    ])
)

### Interactive unfolding explorer / Explorador interactivo de unfolding

Choose the mode to place on the rows.

The tensor values stay the same; only their arrangement changes.

> 🇪🇸 Elige qué modo colocar en las filas. Los valores del tensor permanecen; solo cambia su organización.

In [ ]:
unfold_mode = widgets.ToggleButtons(
    options=[
        ("Mode 0 · Pickup / Origen", 0),
        ("Mode 1 · Dropoff / Destino", 1),
        ("Mode 2 · Hour / Hora", 2),
    ],
    value=2,
    description="Mode / Modo:",
    style={"description_width": "95px"},
)

def explore_unfolding(axis):
    M = unfold(
        T,
        axis,
    )

    mode_labels = {
        0: (
            "pickup borough",
            "distrito de origen",
        ),
        1: (
            "dropoff borough",
            "distrito de destino",
        ),
        2: (
            "hour",
            "hora",
        ),
    }

    en, es = mode_labels[axis]

    fig, ax = plt.subplots(
        figsize=(9, 3.8),
        constrained_layout=True,
    )

    im = ax.imshow(
        M,
        aspect="auto",
        cmap="viridis",
    )

    ax.set_title(
        f"Mode-{axis} unfolding / Unfolding modo {axis}\n"
        f"rows = {en} / filas = {es}"
    )
    ax.set_xlabel(
        "all other modes flattened / "
        "otros modos aplanados"
    )
    ax.set_ylabel(
        f"{en} / {es}"
    )

    fig.colorbar(
        im,
        ax=ax,
        label="trip count / viajes",
    )

    plt.show()

    print("Tensor shape / Forma tensor:", T.shape)
    print("Matrix shape / Forma matriz:", M.shape)
    print(
        "Same number of entries / "
        "Mismo número de entradas:",
        M.size == T.size,
    )
    print()
    print("EN: the selected mode becomes the row axis.")
    print("ES: el modo seleccionado se convierte en el eje de filas.")

unfold_output = widgets.interactive_output(
    explore_unfolding,
    {"axis": unfold_mode},
)

display(
    widgets.VBox([
        unfold_mode,
        unfold_output,
    ])
)

<details>
<summary><strong>Why do we unfold? / ¿Por qué hacemos unfolding?</strong></summary>

SVD works on matrices.

A tensor has more than two axes.

So HOSVD creates one matrix view per mode:

- pickup unfolding → study origin patterns;
- dropoff unfolding → study destination patterns;
- hour unfolding → study temporal patterns.

No trip count is lost by the unfolding itself.

> 🇪🇸 SVD trabaja con matrices, por lo que HOSVD crea una vista matricial para cada modo.
>
> El unfolding no comprime todavía; solo reorganiza las entradas para poder estudiar un modo a la vez.

</details>

## 10.2 HOSVD — compress each mode separately / HOSVD — comprime cada modo por separado

Think of HOSVD as creating **three small dictionaries of patterns**.

### Pickup factor matrix `U₁`
Columns describe important origin patterns.

### Dropoff factor matrix `U₂`
Columns describe important destination patterns.

### Hour factor matrix `U₃`
Columns describe important temporal patterns.

If we retain ranks:

`(r₁, r₂, r₃)`

then the factor shapes are:

- `U₁`: `(pickup_size, r₁)`
- `U₂`: `(dropoff_size, r₂)`
- `U₃`: `(24, r₃)`

and the core has:

`(r₁, r₂, r₃)`

### What is the core? / ¿Qué es el núcleo?

The core is not another raw taxi table.

It stores how the retained pickup, dropoff, and hour patterns interact.

> 🇪🇸 Piensa en HOSVD como tres pequeños diccionarios de patrones: uno para origen, uno para destino y uno para tiempo.
>
> El tensor núcleo describe cómo interactúan esos patrones retenidos.

### Connect to Notebook 06 / Conexión con el Notebook 06

The Tucker core is computed with:

`ijk,ia,jb,kc->abc`

Read the disappearing indices:

- `i` = pickup → contracted;
- `j` = dropoff → contracted;
- `k` = hour → contracted.

The new retained coordinates:

- `a`,
- `b`,
- `c`

become the core axes.

Reconstruction reverses the map:

`abc,ia,jb,kc->ijk`

> 🇪🇸 En la construcción del núcleo, `i`, `j` y `k` desaparecen porque se contraen. Los índices reducidos `a`, `b` y `c` forman los ejes del núcleo.
>
> La reconstrucción realiza el camino inverso para volver a los ejes originales `i,j,k`.

## Exercise 2 — Tucker/HOSVD compression / Ejercicio 2 — compresión Tucker/HOSVD

Start with:

`ranks = (2, 2, 3)`

Then:

1. take the first retained SVD directions from each mode;
2. contract the tensor into a small core;
3. reconstruct the tensor;
4. measure reconstruction error;
5. count how many numbers the Tucker representation stores.

### Relative Frobenius error / Error relativo de Frobenius

`||T - T_hat|| / ||T||`

- `0` = exact reconstruction;
- larger values = more information was lost.

### Storage ratio / Razón de almacenamiento

We report:

`original numbers / Tucker numbers`

- greater than `1` → fewer numbers are stored;
- less than `1` → this Tucker configuration actually stores more numbers than the original tensor.

> 🇪🇸 Comienza con rangos `(2,2,3)`, construye el núcleo, reconstruye el tensor y mide tanto el error como el almacenamiento.
>
> Una razón de almacenamiento mayor que `1` significa que la representación Tucker usa menos números que el tensor original.

In [ ]:
# TODO 2 / TAREA 2
#
# EN:
# 1. Compute one SVD basis for each unfolding.
# 2. Keep ranks (2, 2, 3).
# 3. Build the Tucker core with:
#       'ijk,ia,jb,kc->abc'
# 4. Reconstruct with:
#       'abc,ia,jb,kc->ijk'
# 5. Compute relative Frobenius error.
# 6. Count original and Tucker stored numbers.
#
# ES:
# 1. Calcula una base SVD para cada unfolding.
# 2. Conserva rangos (2, 2, 3).
# 3. Construye el núcleo Tucker con einsum.
# 4. Reconstruye el tensor con einsum.
# 5. Calcula el error relativo de Frobenius.
# 6. Cuenta los números almacenados en el original y en Tucker.

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

bases = hosvd_bases(T)

def tucker_from_ranks(
    T,
    bases,
    ranks,
):
    factors = [
        bases[axis][
            :,
            :ranks[axis],
        ]
        for axis in range(3)
    ]

    core = np.einsum(
        "ijk,ia,jb,kc->abc",
        T,
        factors[0],
        factors[1],
        factors[2],
    )

    recon = np.einsum(
        "abc,ia,jb,kc->ijk",
        core,
        factors[0],
        factors[1],
        factors[2],
    )

    error = (
        np.linalg.norm(T - recon)
        / np.linalg.norm(T)
    )

    factor_numbers = sum(
        factor.size
        for factor in factors
    )

    tucker_numbers = (
        core.size
        + factor_numbers
    )

    storage_ratio = (
        T.size
        / tucker_numbers
    )

    storage_percent = (
        100
        * tucker_numbers
        / T.size
    )

    return (
        core,
        factors,
        recon,
        error,
        tucker_numbers,
        storage_ratio,
        storage_percent,
    )

default_ranks = (
    min(2, bases[0].shape[1]),
    min(2, bases[1].shape[1]),
    min(3, bases[2].shape[1]),
)

(
    core,
    factors,
    recon,
    error,
    tucker_numbers,
    storage_ratio,
    storage_percent,
) = tucker_from_ranks(
    T,
    bases,
    default_ranks,
)

print(
    "Ranks / Rangos:",
    default_ranks,
)
print(
    "Factor shapes / Formas de factores:",
    [u.shape for u in factors],
)
print(
    "Core shape / Forma del núcleo:",
    core.shape,
)
print(
    "Original stored numbers / Números originales:",
    T.size,
)
print(
    "Tucker stored numbers / Números Tucker:",
    tucker_numbers,
)
print(
    "Storage percent / Porcentaje de almacenamiento:",
    f"{storage_percent:.1f}%",
)
print(
    "Original/Tucker ratio / Razón original/Tucker:",
    f"{storage_ratio:.2f}x",
)
print(
    "Relative error / Error relativo:",
    f"{error:.4f}",
)

### Interactive Tucker rank explorer / Explorador interactivo de rangos Tucker

Move the three ranks independently.

Also choose an hour to compare:

- real origin→destination matrix;
- Tucker reconstruction;
- absolute reconstruction error.

Watch what happens when you reduce one mode more aggressively than the others.

> 🇪🇸 Mueve independientemente los tres rangos y elige una hora.
>
> Compara la matriz real, la reconstrucción Tucker y el error absoluto.
>
> Observa qué ocurre cuando comprimes un modo mucho más que los demás.

In [ ]:
pickup_rank = widgets.IntSlider(
    value=default_ranks[0],
    min=1,
    max=bases[0].shape[1],
    step=1,
    description="Pickup rank / Rango origen:",
    continuous_update=False,
    style={"description_width": "170px"},
)

dropoff_rank = widgets.IntSlider(
    value=default_ranks[1],
    min=1,
    max=bases[1].shape[1],
    step=1,
    description="Dropoff rank / Rango destino:",
    continuous_update=False,
    style={"description_width": "180px"},
)

hour_rank = widgets.IntSlider(
    value=default_ranks[2],
    min=1,
    max=bases[2].shape[1],
    step=1,
    description="Hour rank / Rango hora:",
    continuous_update=False,
    style={"description_width": "155px"},
)

compare_hour = widgets.IntSlider(
    value=busiest_hour,
    min=0,
    max=23,
    step=1,
    description="Hour / Hora:",
    continuous_update=False,
    style={"description_width": "95px"},
)

def explore_tucker(
    r_pickup,
    r_dropoff,
    r_hour,
    hour,
):
    ranks = (
        r_pickup,
        r_dropoff,
        r_hour,
    )

    (
        live_core,
        live_factors,
        live_recon,
        live_error,
        live_numbers,
        live_ratio,
        live_percent,
    ) = tucker_from_ranks(
        T,
        bases,
        ranks,
    )

    real_hour = T[:, :, hour]
    reconstructed_hour = live_recon[:, :, hour]
    absolute_error = np.abs(
        real_hour - reconstructed_hour
    )

    vmax = max(
        real_hour.max(),
        reconstructed_hour.max(),
        1,
    )

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(13.5, 4.2),
        constrained_layout=True,
    )

    im0 = axes[0].imshow(
        real_hour,
        cmap="viridis",
        aspect="auto",
        vmin=0,
        vmax=vmax,
    )

    axes[0].set_title(
        f"Real counts · hour {hour}\n"
        f"Conteos reales · hora {hour}",
        fontsize=10,
        pad=10,
    )

    axes[1].imshow(
        reconstructed_hour,
        cmap="viridis",
        aspect="auto",
        vmin=0,
        vmax=vmax,
    )

    axes[1].set_title(
        "Tucker reconstruction\n"
        "Reconstrucción Tucker",
        fontsize=10,
        pad=10,
    )

    im2 = axes[2].imshow(
        absolute_error,
        cmap="magma",
        aspect="auto",
        vmin=0,
    )

    axes[2].set_title(
        "Absolute error\n"
        "Error absoluto",
        fontsize=10,
        pad=10,
    )

    for ax in axes:
        ax.set_xticks(
            range(len(dropoff_names))
        )
        ax.set_xticklabels(
            dropoff_names,
            rotation=35,
            ha="right",
            fontsize=7,
        )
        ax.set_yticks(
            range(len(pickup_names))
        )
        ax.set_yticklabels(
            pickup_names,
            fontsize=7,
        )
        ax.set_xlabel(
            "dropoff / destino"
        )
        ax.set_ylabel(
            "pickup / origen"
        )

    fig.colorbar(
        im0,
        ax=axes[:2],
        shrink=0.8,
        label="trip count / viajes",
    )

    fig.colorbar(
        im2,
        ax=axes[2],
        shrink=0.8,
        label="absolute error / error absoluto",
    )

    plt.show()

    print("Ranks / Rangos:", ranks)
    print(
        "Core / Núcleo:",
        live_core.shape,
    )
    print(
        "Relative error / Error relativo:",
        f"{live_error:.4f}",
    )
    print(
        "Stored numbers / Números almacenados:",
        f"{live_numbers} of/de {T.size}",
    )
    print(
        "Storage percent / Porcentaje almacenamiento:",
        f"{live_percent:.1f}%",
    )
    print(
        "Original/Tucker ratio / Razón original/Tucker:",
        f"{live_ratio:.2f}x",
    )
    print()

    if live_ratio > 1:
        print("EN: this configuration stores fewer numbers than the original tensor.")
        print("ES: esta configuración almacena menos números que el tensor original.")
    else:
        print("EN: this configuration is not storage-compressing; the factors + core use at least as many numbers as T.")
        print("ES: esta configuración no comprime almacenamiento; factores + núcleo usan al menos tantos números como T.")

rank_output = widgets.interactive_output(
    explore_tucker,
    {
        "r_pickup": pickup_rank,
        "r_dropoff": dropoff_rank,
        "r_hour": hour_rank,
        "hour": compare_hour,
    },
)

display(
    widgets.VBox([
        pickup_rank,
        dropoff_rank,
        hour_rank,
        compare_hour,
        rank_output,
    ])
)

### Interactive core-tensor explorer / Explorador interactivo del tensor núcleo

The core has shape:

`(r_pickup, r_dropoff, r_hour)`

Choose one retained **hour component** of the default core.

The heatmap shows how retained pickup and dropoff patterns interact inside that core slice.

> 🇪🇸 El núcleo tiene forma `(r_origen, r_destino, r_hora)`.
>
> Elige un componente horario retenido. El mapa muestra cómo interactúan los patrones reducidos de origen y destino dentro de ese corte del núcleo.

In [ ]:
core_hour_component = widgets.IntSlider(
    value=0,
    min=0,
    max=core.shape[2] - 1,
    step=1,
    description="Core hour c / Hora núcleo c:",
    continuous_update=False,
    style={"description_width": "160px"},
)

def explore_core(c):
    core_slice = core[:, :, c]

    fig, ax = plt.subplots(
        figsize=(5.2, 4.2),
        constrained_layout=True,
    )

    vmax = max(
        np.abs(core_slice).max(),
        1e-12,
    )

    im = ax.imshow(
        core_slice,
        cmap="coolwarm",
        vmin=-vmax,
        vmax=vmax,
        aspect="auto",
    )

    ax.set_xlabel(
        "retained dropoff component b / "
        "componente destino b"
    )
    ax.set_ylabel(
        "retained pickup component a / "
        "componente origen a"
    )

    ax.set_title(
        f"Core slice c={c}\n"
        f"Corte del núcleo c={c}"
    )

    ax.set_xticks(
        range(core_slice.shape[1])
    )
    ax.set_yticks(
        range(core_slice.shape[0])
    )

    for i in range(core_slice.shape[0]):
        for j in range(core_slice.shape[1]):
            ax.text(
                j,
                i,
                f"{core_slice[i, j]:.1f}",
                ha="center",
                va="center",
                fontsize=8,
            )

    fig.colorbar(
        im,
        ax=ax,
        label="core coefficient / coeficiente núcleo",
    )

    plt.show()

    print("Core shape / Forma del núcleo:", core.shape)
    print("Selected c / c seleccionado:", c)
    print("EN: the core coordinates are compressed pattern coordinates, not raw borough or hour labels.")
    print("ES: las coordenadas del núcleo son coordenadas de patrones comprimidos, no etiquetas directas de distrito u hora.")

core_output = widgets.interactive_output(
    explore_core,
    {"c": core_hour_component},
)

display(
    widgets.VBox([
        core_hour_component,
        core_output,
    ])
)

### What does “rank 2” mean here? / ¿Qué significa “rango 2” aquí?

If pickup rank is `2`, Tucker is **not saying there are only two pickup boroughs**.

It says:

> **We keep two learned directions to represent variation across the pickup mode.**

Likewise:

- hour rank `3` does not mean only three hours exist;
- it means we keep three learned temporal directions.

> 🇪🇸 Si el rango de origen es `2`, Tucker no afirma que existan solo dos distritos.
>
> Significa que conservamos **dos direcciones aprendidas** para representar la variación del modo de origen.

### Rank trade-off explorer / Explorador del compromiso de rango

To make the trade-off visible, vary only the **hour rank** while keeping pickup and dropoff ranks fixed at their default values.

The curve shows:

- reconstruction error;
- percentage of original storage used by the Tucker representation.

> 🇪🇸 Para visualizar el compromiso, cambia solo el **rango horario** manteniendo fijos los rangos de origen y destino.
>
> La gráfica muestra el error de reconstrucción y el porcentaje de almacenamiento usado.

In [ ]:
tradeoff_hour_rank = widgets.IntSlider(
    value=default_ranks[2],
    min=1,
    max=bases[2].shape[1],
    step=1,
    description="Hour rank / Rango hora:",
    continuous_update=False,
    style={"description_width": "155px"},
)

hour_rank_grid = np.arange(
    1,
    bases[2].shape[1] + 1,
)

hour_rank_errors = []
hour_rank_storage = []

for r_hour in hour_rank_grid:
    (
        _,
        _,
        _,
        e,
        _,
        _,
        percent,
    ) = tucker_from_ranks(
        T,
        bases,
        (
            default_ranks[0],
            default_ranks[1],
            int(r_hour),
        ),
    )

    hour_rank_errors.append(e)
    hour_rank_storage.append(percent)

hour_rank_errors = np.asarray(
    hour_rank_errors
)

hour_rank_storage = np.asarray(
    hour_rank_storage
)

def explore_tradeoff(r_hour):
    idx = r_hour - 1

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(10.5, 3.8),
        constrained_layout=True,
    )

    axes[0].plot(
        hour_rank_grid,
        hour_rank_errors,
        marker="o",
    )
    axes[0].scatter(
        [r_hour],
        [hour_rank_errors[idx]],
        s=90,
        zorder=5,
    )
    axes[0].set_xlabel(
        "hour rank / rango hora"
    )
    axes[0].set_ylabel(
        "relative error / error relativo"
    )
    axes[0].set_title(
        "Reconstruction error\n"
        "Error de reconstrucción"
    )

    axes[1].plot(
        hour_rank_grid,
        hour_rank_storage,
        marker="o",
    )
    axes[1].scatter(
        [r_hour],
        [hour_rank_storage[idx]],
        s=90,
        zorder=5,
    )
    axes[1].axhline(
        100,
        linestyle="--",
        linewidth=1,
    )
    axes[1].set_xlabel(
        "hour rank / rango hora"
    )
    axes[1].set_ylabel(
        "% of original storage / "
        "% almacenamiento original"
    )
    axes[1].set_title(
        "Storage cost\n"
        "Costo de almacenamiento"
    )

    plt.show()

    print(
        "Selected hour rank / Rango horario seleccionado:",
        r_hour,
    )
    print(
        "Relative error / Error relativo:",
        f"{hour_rank_errors[idx]:.4f}",
    )
    print(
        "Storage / Almacenamiento:",
        f"{hour_rank_storage[idx]:.1f}% of/de original",
    )
    print()
    print("EN: increasing rank usually reduces approximation error but stores more parameters.")
    print("ES: aumentar el rango suele reducir el error de aproximación, pero almacena más parámetros.")

tradeoff_output = widgets.interactive_output(
    explore_tradeoff,
    {"r_hour": tradeoff_hour_rank},
)

display(
    widgets.VBox([
        tradeoff_hour_rank,
        tradeoff_output,
    ])
)

<details>
<summary><strong>What did Exercise 2 show? / ¿Qué mostró el Ejercicio 2?</strong></summary>

Tucker gives each mode its own compression budget.

That is different from forcing all modes to share one rank.

The ranks can reflect different structural complexity:

- pickup may need one number of directions;
- dropoff may need another;
- hour may need another.

The core then connects those retained directions.

> 🇪🇸 Tucker permite asignar un presupuesto de compresión diferente a cada modo.
>
> Los rangos no tienen que ser iguales porque origen, destino y hora pueden tener complejidades distintas.

</details>

## Exercise 3 — what did the temporal factors learn? / Ejercicio 3 — ¿qué aprendieron los factores temporales?

The hour basis:

`U₃.shape = (24, number_of_hour_components)`

has:

- one row per real hour;
- one column per learned temporal direction.

### Important interpretation rule / Regla importante de interpretación

The sign of an SVD singular vector is arbitrary.

A component can be multiplied by `-1` and still represent the same direction.

So when asking **where** a component is strongest, it is often safer to inspect:

`abs(component)`

### Do not over-interpret / No sobreinterpretes

A temporal factor is an exploratory learned pattern.

It is not automatically a named concept such as “rush hour” unless the evidence supports that interpretation.

> 🇪🇸 La base horaria tiene una fila por hora y una columna por dirección temporal aprendida.
>
> El signo de un vector singular es arbitrario, por lo que para localizar dónde un componente es más fuerte suele ser útil observar su magnitud absoluta.
>
> Un factor temporal es un patrón exploratorio; no debe recibir automáticamente una etiqueta causal como “hora pico”.

In [ ]:
# TODO 3 / TAREA 3
#
# EN:
# 1. Use the hour-mode HOSVD basis.
# 2. Inspect temporal component 1.
# 3. Find the hour of its largest absolute loading.
# 4. Compare it with raw hourly taxi counts.
# 5. Repeat for another component.
# 6. Describe the shape of the pattern without inventing a causal label.
#
# ES:
# 1. Usa la base HOSVD del modo hora.
# 2. Inspecciona el componente temporal 1.
# 3. Encuentra la hora de mayor carga absoluta.
# 4. Compárala con los conteos reales por hora.
# 5. Repite con otro componente.
# 6. Describe la forma del patrón sin inventar una etiqueta causal.

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

hour_basis = bases[2]
raw_hour_counts = T.sum(
    axis=(0, 1)
)

first_component = hour_basis[:, 0]

first_peak = int(
    np.argmax(
        np.abs(first_component)
    )
)

print(
    "Hour basis / Base horaria:",
    hour_basis.shape,
)
print(
    "First component peak / "
    "Pico del primer componente:",
    first_peak,
)
print(
    "Raw busiest hour / "
    "Hora real más ocupada:",
    busiest_hour,
)
print()
print("EN: the two hours do not have to be identical; an SVD component describes a direction of variation, not simply the raw total.")
print("ES: las dos horas no tienen que coincidir; un componente SVD describe una dirección de variación, no simplemente el total bruto.")

### Interactive temporal-factor explorer / Explorador interactivo de factores temporales

Choose a learned temporal component.

The graph compares:

- normalized real hourly trip totals;
- normalized absolute temporal-factor loading.

> 🇪🇸 Elige un componente temporal aprendido.
>
> La gráfica compara los conteos horarios reales normalizados con la magnitud normalizada del factor temporal.

In [ ]:
component_slider = widgets.IntSlider(
    value=1,
    min=1,
    max=hour_basis.shape[1],
    step=1,
    description="Component / Componente:",
    continuous_update=False,
    style={"description_width": "145px"},
)

def explore_hour_factor(component):
    idx = component - 1

    factor = hour_basis[:, idx]

    peak = int(
        np.argmax(
            np.abs(factor)
        )
    )

    raw_scaled = (
        raw_hour_counts
        / raw_hour_counts.max()
    )

    factor_scaled = np.abs(
        factor
    )

    factor_scaled = (
        factor_scaled
        / factor_scaled.max()
    )

    fig, ax = plt.subplots(
        figsize=(8.8, 4.0),
        constrained_layout=True,
    )

    ax.plot(
        range(24),
        raw_scaled,
        marker="o",
        label="raw hourly trips / viajes reales",
    )

    ax.plot(
        range(24),
        factor_scaled,
        marker="o",
        label=(
            f"|temporal factor {component}| / "
            f"|factor temporal {component}|"
        ),
    )

    ax.axvline(
        peak,
        linestyle="--",
        linewidth=1,
        label=(
            f"factor peak / pico factor = {peak}"
        ),
    )

    ax.set_xticks(
        range(0, 24, 2)
    )

    ax.set_xlabel(
        "hour / hora"
    )

    ax.set_ylabel(
        "scaled magnitude / "
        "magnitud escalada"
    )

    ax.set_title(
        f"Raw activity vs temporal component {component}\n"
        f"Actividad real vs componente temporal {component}"
    )

    ax.legend(
        fontsize=8
    )

    plt.show()

    print(
        "Component / Componente:",
        component,
    )
    print(
        "Peak absolute loading / "
        "Pico de carga absoluta:",
        peak,
    )
    print(
        "Raw busiest hour / "
        "Hora real más ocupada:",
        busiest_hour,
    )
    print()
    print("EN: compare the shape of the curves; do not assume the factor is just the raw traffic total.")
    print("ES: compara la forma de las curvas; no supongas que el factor es simplemente el total bruto de tráfico.")

factor_output = widgets.interactive_output(
    explore_hour_factor,
    {"component": component_slider},
)

display(
    widgets.VBox([
        component_slider,
        factor_output,
    ])
)

### Explore factors from any mode / Explora factores de cualquier modo

Choose:

- pickup / origen;
- dropoff / destino;
- hour / hora.

Then choose a component.

The plot shows the loadings of that learned direction across the original mode labels.

> 🇪🇸 Elige origen, destino u hora y después un componente. La gráfica muestra las cargas de esa dirección aprendida sobre las etiquetas originales del modo.

In [ ]:
factor_mode = widgets.ToggleButtons(
    options=[
        ("Pickup / Origen", 0),
        ("Dropoff / Destino", 1),
        ("Hour / Hora", 2),
    ],
    value=2,
    description="Mode / Modo:",
    style={"description_width": "95px"},
)

factor_component = widgets.IntSlider(
    value=1,
    min=1,
    max=max(
        basis.shape[1]
        for basis in bases
    ),
    step=1,
    description="Component / Componente:",
    continuous_update=False,
    style={"description_width": "145px"},
)

def explore_mode_factor(axis, component):
    basis = bases[axis]

    available = basis.shape[1]

    idx = min(
        component,
        available,
    ) - 1

    values = basis[:, idx]

    if axis == 0:
        labels = pickup_names
        mode_en = "pickup"
        mode_es = "origen"
    elif axis == 1:
        labels = dropoff_names
        mode_en = "dropoff"
        mode_es = "destino"
    else:
        labels = list(range(24))
        mode_en = "hour"
        mode_es = "hora"

    fig, ax = plt.subplots(
        figsize=(8.8, 3.8),
        constrained_layout=True,
    )

    ax.bar(
        np.arange(len(values)),
        values,
    )

    ax.set_xticks(
        np.arange(len(values))
    )

    ax.set_xticklabels(
        labels,
        rotation=35 if axis != 2 else 0,
        ha="right" if axis != 2 else "center",
        fontsize=8,
    )

    ax.set_ylabel(
        "factor loading / carga del factor"
    )

    ax.set_title(
        f"{mode_en} factor {idx + 1} / "
        f"factor de {mode_es} {idx + 1}"
    )

    plt.show()

    print(
        "Mode / Modo:",
        axis,
        f"({mode_en} / {mode_es})",
    )
    print(
        "Available components / Componentes disponibles:",
        available,
    )
    print(
        "Displayed component / Componente mostrado:",
        idx + 1,
    )
    print()
    print("EN: positive and negative signs describe an SVD direction; the whole vector pattern matters more than a single sign.")
    print("ES: los signos positivos y negativos describen una dirección SVD; importa más el patrón completo del vector que un signo aislado.")

all_factor_output = widgets.interactive_output(
    explore_mode_factor,
    {
        "axis": factor_mode,
        "component": factor_component,
    },
)

display(
    widgets.VBox([
        factor_mode,
        factor_component,
        all_factor_output,
    ])
)

## Quick reasoning challenge / Reto rápido de razonamiento

Choose a statement and decide whether it is correct.

> 🇪🇸 Elige una afirmación y comprueba si es correcta.

In [ ]:
tucker_question = widgets.Dropdown(
    options=[
        (
            "Unfolding deletes data / "
            "Unfolding elimina datos",
            "unfold",
        ),
        (
            "Rank (2,2,3) means only 2 pickups, 2 dropoffs, 3 hours exist / "
            "Rango (2,2,3) significa que solo existen 2 orígenes, 2 destinos y 3 horas",
            "rank",
        ),
        (
            "Higher rank usually stores more and reconstructs better / "
            "Mayor rango suele almacenar más y reconstruir mejor",
            "tradeoff",
        ),
        (
            "Temporal factor sign is uniquely meaningful / "
            "El signo del factor temporal tiene significado único",
            "sign",
        ),
    ],
    value="tradeoff",
    description="Statement / Afirmación:",
    style={"description_width": "150px"},
)

def explain_tucker_question(question):
    answers = {
        "unfold": (
            "FALSE",
            "Unfolding rearranges entries but does not delete them.",
            "FALSO",
            "Unfolding reorganiza las entradas, pero no las elimina.",
        ),
        "rank": (
            "FALSE",
            "The ranks count retained learned directions, not the number of real labels that exist.",
            "FALSO",
            "Los rangos cuentan direcciones aprendidas conservadas, no cuántas etiquetas reales existen.",
        ),
        "tradeoff": (
            "GENERALLY TRUE",
            "Keeping more directions usually lowers approximation error while increasing storage.",
            "GENERALMENTE VERDADERO",
            "Conservar más direcciones suele reducir el error de aproximación y aumentar el almacenamiento.",
        ),
        "sign": (
            "FALSE",
            "SVD vector signs can flip without changing the represented direction.",
            "FALSO",
            "Los signos de los vectores SVD pueden invertirse sin cambiar la dirección representada.",
        ),
    }

    en_state, en, es_state, es = answers[question]

    print("EN:", en_state)
    print("   ", en)
    print("ES:", es_state)
    print("   ", es)

question_output = widgets.interactive_output(
    explain_tucker_question,
    {"question": tucker_question},
)

display(
    widgets.VBox([
        tucker_question,
        question_output,
    ])
)

## What just happened / Qué acaba de pasar

You built a complete Tucker/HOSVD pipeline from real observations.

### 1. Real tensor construction / Construcción del tensor real

A flat taxi table became:

`pickup × dropoff × hour`

Each tensor entry remained connected to a concrete real-world meaning.

### 2. Unfolding / Desplegado

The same entries were rearranged into three matrix views.

Each unfolding put one semantic mode on the rows.

### 3. HOSVD

SVD learned one basis for each mode:

- pickup patterns;
- dropoff patterns;
- temporal patterns.

### 4. Tucker core / Núcleo Tucker

`ijk,ia,jb,kc->abc`

compressed the original coordinates into retained pattern coordinates.

The core described how those mode-specific patterns interact.

### 5. Rank trade-off / Compromiso de rango

Reducing rank stored fewer directions but increased approximation error.

Increasing rank usually improved reconstruction but required more stored numbers.

### 6. Interpretation / Interpretación

The temporal factors could be compared directly with real hourly activity, but their sign is arbitrary and their meaning should be interpreted cautiously.

### The sentence to remember / La frase para recordar

> **Tucker compresses a tensor by learning a separate basis for each mode and a small core that describes how those learned patterns interact.**

> 🇪🇸
>
> **Tucker comprime un tensor aprendiendo una base separada para cada modo y un núcleo pequeño que describe cómo interactúan esos patrones aprendidos.**

### Final self-check / Autoevaluación final

If Tucker uses ranks:

`(2, 2, 3)`

does that mean the original tensor now has only seven real categories?

**No.**

It means the model uses:

- 2 learned pickup directions;
- 2 learned dropoff directions;
- 3 learned temporal directions;

to approximate the full original tensor.

> 🇪🇸 Un rango Tucker `(2,2,3)` no elimina las categorías reales originales.
>
> Indica cuántas **direcciones aprendidas** se conservan en cada modo para aproximar el tensor completo.

---

## Time for Kahoot 🎯 / Hora de Kahoot 🎯

**Kahoot 3 — Convolution & Tensor Decompositions / Convolución y descomposiciones tensoriales**  
6 questions / 6 preguntas · about 5 minutes / unos 5 minutos.

Join at **kahoot.it** with the PIN on the facilitator's screen.

> 🇪🇸 Entra a **kahoot.it** con el PIN que aparece en la pantalla del facilitador.

- [Quiz details and facilitator notes](https://project-delphi.github.io/tensors-workshop/kahoot.html#quiz-3)
- [Import file (`.xlsx`)](https://github.com/project-delphi/tensors-workshop/blob/main/kahoot/kahoot_quiz_3_convolution_decompositions.xlsx)

Next / Siguiente: **11 · Wrap-up and take-homes / Cierre y ejercicios para casa** — [open in Colab](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/11-wrap-up-and-take-homes.ipynb).

[← Workshop site / Sitio del taller](https://project-delphi.github.io/tensors-workshop/) · [All notebooks / Todos los notebooks](https://project-delphi.github.io/tensors-workshop/notebooks.html) · [Handbook / Manual](https://project-delphi.github.io/tensors-workshop/tensors_workshop_plan_with_quizzes.html)